# RAG Pipeline over a PDF

End-to-end Retrieval-Augmented Generation: **load PDF → chunk → embed → store in a vector index → retrieve → generate an answer.**

**Before you run:** avoid putting Confidential/Strictly Confidential or personal data into a third-party LLM. Use non-sensitive documents unless your setup is approved for the classification involved.

## 1. Install dependencies

In [ ]:
%pip install -q pypdf sentence-transformers faiss-cpu numpy anthropic

## 2. Configuration

In [ ]:
PDF_PATH = "document.pdf"          # <-- set to your PDF path
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 800                   # characters per chunk
CHUNK_OVERLAP = 120                # overlap between chunks
TOP_K = 4                          # chunks retrieved per query
LLM_MODEL = "claude-sonnet-4-6"

import os
# Set your key in the environment before running, e.g. export ANTHROPIC_API_KEY=...
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

## 3. Load & extract text from the PDF

In [ ]:
from pypdf import PdfReader

def load_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"page": i + 1, "text": text})
    return pages

pages = load_pdf(PDF_PATH)
print(f"Loaded {len(pages)} pages with text.")

## 4. Chunk the text

In [ ]:
def chunk_text(text, size, overlap):
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

documents = []
for p in pages:
    for c in chunk_text(p["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        if c.strip():
            documents.append({"page": p["page"], "text": c.strip()})

print(f"Created {len(documents)} chunks.")

## 5. Embed chunks & build a FAISS index

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product on normalized = cosine
index.add(embeddings)
print(f"Indexed {index.ntotal} vectors, dim={embeddings.shape[1]}.")

## 6. Retrieval

In [ ]:
def retrieve(query, k=TOP_K):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    results = []
    for score, i in zip(scores[0], idx[0]):
        d = documents[i]
        results.append({"score": float(score), "page": d["page"], "text": d["text"]})
    return results

for r in retrieve("What is this document about?"):
    print(f"[p.{r['page']} | {r['score']:.3f}] {r['text'][:120]}...")

## 7. Generation with retrieved context

In [ ]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from env

def build_context(chunks):
    return "\n\n".join(f"[Source: page {c['page']}]\n{c['text']}" for c in chunks)

def answer(query, k=TOP_K):
    chunks = retrieve(query, k)
    context = build_context(chunks)
    prompt = (
        "Answer the question using only the context below. "
        "Cite the page numbers you used. If the answer is not in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text, chunks

response, sources = answer("Summarize the main points of this document.")
print(response)
print("\n--- Sources ---")
for s in sources:
    print(f"page {s['page']} (score {s['score']:.3f})")

## 8. Ask your own questions

In [ ]:
query = "Type your question here"
response, sources = answer(query)
print(response)